# 05 — GenAI Advice Generation

Integrate an open-source LLM (via Databricks Foundation Model APIs or Ollama)
to generate natural language financial summaries and personalized advice.

In [0]:
# Environment-aware setup: auto-installs deps only on Databricks
import os

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', 'langchain', 'langchain-community', 'pandas>=2.0'])
    dbutils.library.restartPython()  # noqa: F821
else:
    print("Running locally — skipping Databricks pip install. "
          "Make sure you've run: pip install -r requirements.txt")

In [0]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.data_loader import load_data, preprocess, get_user_data, get_summary_stats
from src.trend_detection import monthly_trends, detect_risky_patterns, financial_health_score
from src.scenario_simulation import simulate_savings_increase
from src.genai_interface import get_llm, generate_financial_summary, generate_personalized_advice, explain_scenario

## 1. Configure LLM

**Databricks**: Uses Foundation Model APIs (e.g., `databricks/databricks-meta-llama-3-1-70b-instruct`)  
**Local dev**: Set `LLM_MODEL=ollama/llama3.1` and run `ollama serve`

In [0]:
from src.env_utils import is_databricks, default_llm_model

if is_databricks():
    os.environ['LLM_MODEL'] = 'databricks/databricks-meta-llama-3-3-70b-instruct'
else:
    os.environ['LLM_MODEL'] = 'ollama/llama3.1'  # or 'openai/gpt-4'

print(f"Environment: {'Databricks' if is_databricks() else 'Local'}")
print(f"LLM Model: {default_llm_model()}")

llm = get_llm()
print(f'LLM initialized: {type(llm).__name__}')

In [0]:
from src.env_utils import default_data_path

# Load user data — uses DBFS path on Databricks, local path otherwise
data_file = default_data_path() if not os.path.exists('../data/virtual_financial_advisor_data.csv') else '../data/virtual_financial_advisor_data.csv'
df = preprocess(load_data(data_file))
user_df = get_user_data(df, 'user_1')
stats = get_summary_stats(user_df)
trends = monthly_trends(user_df)
risks = detect_risky_patterns(user_df)
health = financial_health_score(user_df)

print('Stats:', stats)
print('Health Score:', health['score'])
print('Risks:', risks)

## 2. Generate Financial Summary

In [0]:
summary_data = {
    **stats,
    'trends': trends.to_string(index=False),
    'risks': '\n'.join(f"- [{r['severity']}] {r['risk']}: {r['detail']}" for r in risks) if risks else 'None',
}

summary = generate_financial_summary(summary_data, llm=llm)
print('=== Financial Summary ===')
print(summary)

## 3. Generate Personalized Advice

In [0]:
scenario = simulate_savings_increase(user_df, increase_pct=10, months=12)

advice_data = {
    **stats,
    'health_score': health['score'],
}

advice = generate_personalized_advice(advice_data, risks, scenario, llm=llm)
print('=== Personalized Advice ===')
print(advice)

## 4. Explain Scenario

In [0]:
explanation = explain_scenario(scenario, llm=llm)
print('=== Scenario Explanation ===')
print(explanation)

## 5. Full Prompt Transparency

Below we show the exact prompt sent to the LLM for the advice generation.

In [0]:
from src.genai_interface import _ADVICE_TEMPLATE

payload = {
    **advice_data,
    'risks': '\n'.join(f"- [{r['severity'].upper()}] {r['risk']}: {r['detail']}" for r in risks) if risks else 'No significant risks.',
    'scenarios': str(scenario),
}

messages = _ADVICE_TEMPLATE.format_messages(**payload)
for msg in messages:
    print(f'--- {msg.type} ---')
    print(msg.content)
    print()